# 2.09 策略梯度：直接优化策略

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.09_policy_gradient.ipynb)


## 为什么要学

前面的方法都是"先学值函数，再从值函数推出策略"（value-based）。**策略梯度**换一种思路：把策略写成带参数的函数 $\pi(a|s;\boldsymbol\theta)$（比如神经网络），**直接对"期望回报"求梯度、做梯度上升**。

好处：

- 天然支持**连续动作**：策略可以输出高斯分布，VLA 可以输出流匹配的动作分布；
- 可以学**随机策略**；
- **VLA + RL 的主流做法（PPO、GRPO、RLinf）都是策略梯度方法**。

对应书中第 9 章。

---

## 1. 目标函数

书中给出两个等价（在一定条件下）的指标：

$$
\bar v_\pi = \sum_s d(s)\,v_\pi(s)=\mathbb{E}_{S\sim d}[v_\pi(S)]\qquad\text{（平均状态值）}
$$

$$
\bar r_\pi = \sum_s d_\pi(s)\,r_\pi(s)\qquad\text{（平均单步奖励）}
$$

对回合制任务，最常用的是"从初始状态出发的期望回报" $J(\boldsymbol\theta)=\mathbb{E}_{\tau\sim\pi_\theta}[G_0]$。目标：$\max_{\boldsymbol\theta}J(\boldsymbol\theta)$。

## 2. 策略梯度定理

**核心结果**：

$$
\boxed{\nabla_{\boldsymbol\theta}J(\boldsymbol\theta)=\mathbb{E}_{S\sim\eta,\,A\sim\pi(\cdot|S;\boldsymbol\theta)}\Big[\nabla_{\boldsymbol\theta}\ln\pi(A|S;\boldsymbol\theta)\,q_\pi(S,A)\Big]}
$$

（$\eta$ 是某个状态分布，对不同的指标略有不同，书中 9.3 节有严格推导。）

### 关键技巧：对数导数（log-derivative trick）

为什么会出现 $\ln\pi$？看最简单的情形：对 $\mathbb{E}_{a\sim\pi_\theta}[f(a)]$ 求导，

$$
\nabla_\theta\sum_a\pi_\theta(a)f(a)=\sum_a\pi_\theta(a)\frac{\nabla_\theta\pi_\theta(a)}{\pi_\theta(a)}f(a)=\mathbb{E}_{a\sim\pi_\theta}\big[\nabla_\theta\ln\pi_\theta(a)\,f(a)\big]
$$

左边"对分布求导"无法直接采样；右边是一个期望，**可以用采样估计**。这也叫"得分函数估计"（score function estimator）。它与 1.05 课的重参数化技巧是估计梯度的两大方法：

- 重参数化：要求 $f$ 可导，方差小（SAC 用）；
- 得分函数：$f$ 可以是不可导的"黑箱"，比如环境奖励（REINFORCE/PPO 用）。

### 直觉

梯度上升 $\boldsymbol\theta\leftarrow\boldsymbol\theta+\alpha\nabla\ln\pi(a|s)\,q(s,a)$：

- $q(s,a)$ 大：**增大**这个动作的概率；
- $q(s,a)$ 小或为负：**减小**它的概率。

本质上就是"**加权的行为克隆**"：做得好的动作多模仿，做得差的少模仿。这个视角在第 8 章非常重要，RECAP、AWR 都是这个思想的变体。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.distributions import Categorical
from vla_ladder.envs import CartPole
from vla_ladder.utils import moving_average, set_seed

### 练习 1：数值验证对数导数技巧

策略是一个 3 类的 softmax：$\pi_\theta=\text{softmax}(\theta)$，奖励 $f=[1, 5, -2]$。分别用(a) 解析求导 (b) 得分函数估计（采样）计算 $\nabla_\theta\mathbb{E}[f(a)]$。

In [ ]:
def score_function_grad(theta, f, n_samples=100000):
    """用 E[∇ ln π(a) f(a)] 估计梯度"""
    # TODO: θ 设为 requires_grad；采样 a ~ Categorical(logits=θ)（采样本身不需要梯度）；
    #       loss = (log_prob(a) * f[a]).mean()；backward 后 θ.grad 就是估计值
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def score_function_grad(theta, f, n_samples=100000):
    """用 E[∇ ln π(a) f(a)] 估计梯度"""
    th = theta.clone().requires_grad_(True)
    dist = Categorical(logits=th)
    a = dist.sample((n_samples,))
    (dist.log_prob(a) * f[a]).mean().backward()
    return th.grad

In [ ]:
# ✅ 自测
theta = torch.tensor([0.2, -0.5, 1.0]); f = torch.tensor([1.0, 5.0, -2.0])
th = theta.clone().requires_grad_(True)
(torch.softmax(th, 0) * f).sum().backward()          # 解析：直接对期望求导
est = score_function_grad(theta, f)
print("解析梯度:", th.grad.numpy().round(3), "  得分函数估计:", est.numpy().round(3))
assert torch.allclose(th.grad, est, atol=0.05)
print("通过 ✅")

---

## 3. REINFORCE：用 MC 回报代替 q

$q_\pi(s_t,a_t)$ 不知道，用 MC 估计：采样完整回合，用**从 $t$ 开始的回报** $g_t=\sum_{k\ge t}\gamma^{k-t}r_{k+1}$ 代替（"reward-to-go"）：

$$
\boldsymbol\theta\leftarrow\boldsymbol\theta+\alpha\sum_t\nabla_{\boldsymbol\theta}\ln\pi(a_t|s_t;\boldsymbol\theta)\,g_t
$$

代码中通常写成最小化"伪损失" $L=-\frac1N\sum_t\ln\pi(a_t|s_t)\,g_t$，让 autograd 帮我们求梯度。

## 4. 基线（baseline）：降低方差

REINFORCE 的梯度估计方差很大。一个关键性质：**减去任何只依赖于状态的基线 $b(s)$，梯度的期望不变**：

$$
\mathbb{E}_{a\sim\pi}\big[\nabla\ln\pi(a|s)\,b(s)\big]=b(s)\sum_a\nabla\pi(a|s)=b(s)\nabla\underbrace{\sum_a\pi(a|s)}_{=1}=0
$$

所以可以用 $(g_t-b(s_t))$ 代替 $g_t$。好的基线能大大减小方差，最自然的选择是 $b(s)=v_\pi(s)$。这时 $g_t-v(s_t)$ 近似**优势函数** $A(s,a)=q(s,a)-v(s)$：这个动作比"平均水平"好多少。下一课的 Actor-Critic 就是这么做的。

最简单的基线：**一个 batch 内回报的均值**（再除以标准差做归一化）。

### 练习 2：REINFORCE 的损失

In [ ]:
def reinforce_loss(logits, actions, returns, normalize=True):
    """logits: (N, A)，actions: (N,)，returns: (N,) 每一步的 reward-to-go。返回标量损失"""
    # TODO:
    # 1) 若 normalize，returns = (returns - mean) / (std + 1e-8)  —— 这就是"均值基线 + 缩放"
    # 2) log_probs = Categorical(logits=logits).log_prob(actions)
    # 3) return -(log_probs * returns).mean()
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def reinforce_loss(logits, actions, returns, normalize=True):
    """logits: (N, A)，actions: (N,)，returns: (N,) 每一步的 reward-to-go。返回标量损失"""
    if normalize:
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    log_probs = Categorical(logits=logits).log_prob(actions)
    return -(log_probs * returns).mean()

In [ ]:
# ✅ 自测：回报高的动作，梯度方向应该让它的概率增大
logits = torch.zeros(2, 2, requires_grad=True)
loss = reinforce_loss(logits, torch.tensor([0, 1]), torch.tensor([10.0, -10.0]))
loss.backward()
lr = 1.0
new_logits = logits - lr * logits.grad          # 梯度下降一步（最小化 loss = 最大化回报）
p = torch.softmax(new_logits, -1)
assert p[0, 0] > 0.5 and p[1, 1] < 0.5          # 好动作概率上升，坏动作概率下降
print("通过 ✅")

### 训练 REINFORCE

In [ ]:
def discounted_returns(rewards, gamma):
    G, out = 0.0, np.zeros(len(rewards))
    for t in reversed(range(len(rewards))):
        G = rewards[t] + gamma * G; out[t] = G
    return out

def train_reinforce(n_updates=80, episodes_per_update=8, gamma=0.99, lr=1e-2, seed=0):
    set_seed(seed)
    env = CartPole()
    policy = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 2))
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    history = []
    for it in range(n_updates):
        obs_buf, act_buf, ret_buf, ep_lens = [], [], [], []
        for ep in range(episodes_per_update):
            obs, _ = env.reset(seed=seed * 10000 + it * 100 + ep); rewards = []; done = False
            while not done:
                with torch.no_grad():
                    a = Categorical(logits=policy(torch.as_tensor(obs))).sample().item()
                obs_buf.append(obs); act_buf.append(a)
                obs, r, term, trunc, _ = env.step(a); rewards.append(r); done = term or trunc
            ret_buf.extend(discounted_returns(rewards, gamma)); ep_lens.append(len(rewards))
        loss = reinforce_loss(policy(torch.as_tensor(np.array(obs_buf))), torch.as_tensor(act_buf),
                              torch.as_tensor(ret_buf, dtype=torch.float32))
        opt.zero_grad(); loss.backward(); opt.step()
        history.append(np.mean(ep_lens))
        if it % 20 == 0:
            print(f"update {it:3d}  平均回合长度 {history[-1]:.1f}")
    return policy, history

policy, hist = train_reinforce()
plt.figure(figsize=(5, 3)); plt.plot(hist); plt.xlabel("update"); plt.ylabel("mean episode return"); plt.title("REINFORCE on CartPole"); plt.show()
print("最后 10 次更新的平均回报:", np.mean(hist[-10:]))
assert np.mean(hist[-10:]) > 150

### 练习 3：亲眼看看基线降低了多少方差

固定一个策略（刚初始化的网络），反复独立地采样小批量数据、估计策略梯度，比较**减基线**与**不减基线**时梯度估计的方差。两者的期望相同（上面证明过），所以方差越小，梯度估计越可靠。

In [ ]:
def pg_grad(policy, obs, acts, rets, baseline):
    """返回策略梯度估计（展平成一个向量）。baseline=True 时用 batch 内回报均值作为基线"""
    policy.zero_grad()
    # TODO: adv = rets - rets.mean() 或 rets；loss = -(log_prob * adv).mean()；backward；
    #       把所有参数的 .grad 展平后拼接返回（torch.cat([p.grad.flatten() for p in policy.parameters()])）
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def pg_grad(policy, obs, acts, rets, baseline):
    """返回策略梯度估计（展平成一个向量）。baseline=True 时用 batch 内回报均值作为基线"""
    policy.zero_grad()
    adv = rets - rets.mean() if baseline else rets
    loss = -(Categorical(logits=policy(obs)).log_prob(acts) * adv).mean()
    loss.backward()
    return torch.cat([p.grad.flatten() for p in policy.parameters()])

In [ ]:
# ✅ 自测
set_seed(1)
pol = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 2))
env = CartPole()
grads = {True: [], False: []}
for trial in range(40):
    obs_buf, act_buf, ret_buf = [], [], []
    for ep in range(2):                                    # 每次只用 2 个回合估计梯度
        o, _ = env.reset(seed=1000 + trial * 2 + ep); rewards, done = [], False
        while not done:
            with torch.no_grad():
                a = Categorical(logits=pol(torch.as_tensor(o))).sample().item()
            obs_buf.append(o); act_buf.append(a)
            o, r, term, trunc, _ = env.step(a); rewards.append(r); done = term or trunc
        ret_buf.extend(discounted_returns(rewards, 0.99))
    O, A_, R_ = torch.as_tensor(np.array(obs_buf)), torch.as_tensor(act_buf), torch.as_tensor(ret_buf, dtype=torch.float32)
    for b in (True, False):
        grads[b].append(pg_grad(pol, O, A_, R_, b))
var = {b: torch.stack(g).var(0).sum().item() for b, g in grads.items()}
print(f"梯度估计的总方差：不减基线 {var[False]:.3f}，减基线 {var[True]:.3f}，降低了 {var[False] / var[True]:.1f} 倍")
assert var[True] < var[False]
print("通过 ✅")

> **解释**：CartPole 每一步奖励都是 +1，所以所有 $g_t$ 都是正数。不减基线时，**每个被采样到的动作的概率都会被增大**，只是增大的程度不同，真正有用的信号（哪个动作更好）被淹没在噪声里。减去均值后，比平均好的动作增大概率，比平均差的减小概率，信号清晰得多。（实际训练中，Adam 的自适应步长会部分掩盖这个差别，但在难任务上基线几乎是必需的。）

---

## 小结

- 策略梯度定理：$\nabla J=\mathbb{E}[\nabla\ln\pi(A|S)\,q_\pi(S,A)]$；对数导数技巧让"对分布求导"变成可以采样的期望。
- REINFORCE：用 MC 回报 $g_t$ 估计 $q$；是 on-policy 的，每次更新都要用当前策略重新采样。
- 基线不改变梯度期望、但能显著降低方差；最好的基线约等于 $v(s)$，于是 $g_t-v(s_t)$ 就是优势。
- 策略梯度 ≈ **按回报加权的行为克隆**。

**思考题**：REINFORCE 每次更新后，之前采的数据就"作废"了（on-policy）。在真机上，一个回合可能要 1 分钟，这意味着什么？有没有办法复用旧数据？（提示：重要性采样，见下一课；PPO 在一定程度上复用数据，见 2.11。）